In [5]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from pathlib import Path

def black_scholes_greeks(S, K, T, r, q, sigma, option_type):
    """
    Calculate Black-Scholes-Merton option price and Greeks
    """
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == 'Call':
        value = S * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
        delta = np.exp(-q * T) * norm.cdf(d1)
        rho = K * T * np.exp(-r * T) * norm.cdf(d2)
    else:  # Put
        value = K * np.exp(-r * T) * norm.cdf(-d2) - S * np.exp(-q * T) * norm.cdf(-d1)
        delta = np.exp(-q * T) * (norm.cdf(d1) - 1)
        rho = -K * T * np.exp(-r * T) * norm.cdf(-d2)

    gamma = np.exp(-q * T) * norm.pdf(d1) / (S * sigma * np.sqrt(T))
    vega = S * np.exp(-q * T) * norm.pdf(d1) * np.sqrt(T)

    term1 = -(S * norm.pdf(d1) * sigma * np.exp(-q * T)) / (2 * np.sqrt(T))
    if option_type == 'Call':
        theta = term1 - r * K * np.exp(-r * T) * norm.cdf(d2) + q * S * np.exp(-q * T) * norm.cdf(d1)
    else:
        theta = term1 + r * K * np.exp(-r * T) * norm.cdf(-d2) - q * S * np.exp(-q * T) * norm.cdf(-d1)

    return value, delta, gamma, vega, rho, theta

# Read data
DATA_DIR = Path.cwd() / "testfiles_" / "data"
CSV_PATH = DATA_DIR / "test12_1.csv"
df = pd.read_csv(CSV_PATH, header=0)

df = df.dropna(subset=['ID'])

results = []
for _, row in df.iterrows():
    S = row['Underlying']
    K = row['Strike']
    T = row['DaysToMaturity'] / row['DayPerYear']
    r = row['RiskFreeRate']
    q = row['DividendRate']
    sigma = row['ImpliedVol']
    option_type = row['Option Type']

    value, delta, gamma, vega, rho, theta = black_scholes_greeks(
        S, K, T, r, q, sigma, option_type
    )

    results.append({
        'ID': int(row['ID']),
        'Value': value,
        'Delta': delta,
        'Gamma': gamma,
        'Vega': vega,
        'Rho': rho,
        'Theta': theta
    })

output_df = pd.DataFrame(results)
print(output_df)

   ID      Value     Delta     Gamma       Vega        Rho      Theta
0   1   3.260824  0.547872  0.053506  14.659079   7.058414 -13.019817
1   2   2.646281 -0.452128  0.053506  14.659079  -6.556032  -8.547471
2   3  22.043329  0.685508  0.008115  35.571438  50.967099  -7.213608
3   4  20.449083 -0.470751  0.009310  40.812329 -73.999110  -5.351164
